### Import Data

In [2]:
import polars as pl
from great_tables import GT
import math

events = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Events.csv",
    schema_overrides={"Player_Id": pl.String},
)

shifts = pl.read_csv("data/2025-10-11.Team.A.@.Team.D.Shifts.csv")

tracking_p1 = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Tracking_P1.csv",
    schema_overrides={"Rink Location Z (Feet)": pl.Float64},
).drop("Player Id")

tracking_p2 = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Tracking_P2.csv",
    schema_overrides={"Rink Location Z (Feet)": pl.Float64},
).drop("Player Id")

tracking_p3 = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Tracking_P3.csv",
    schema_overrides={"Rink Location Z (Feet)": pl.Float64},
).drop("Player Id")


# Convert clock strings ("MM:SS" / "M:SS") to total seconds.
# events.Clock uses zero-padded minutes ("09:30"); tracking "Game Clock" is unpadded ("9:30").
# Casting both to integer seconds makes them directly comparable for joins.
def clock_to_seconds(col):
    parts = pl.col(col).str.split(":")
    return (
        parts.list.get(0).cast(pl.Int64) * 60
        + parts.list.get(1).cast(pl.Int64)
    )


# Add the seconds column and slot it right after its source column.
def add_seconds_after(df, src, new):
    df = df.with_columns(clock_to_seconds(src).alias(new))
    cols = [c for c in df.columns if c != new]
    cols.insert(df.get_column_index(src) + 1, new)
    return df.select(cols)


events = add_seconds_after(events, "Clock", "Clock_Seconds")

tracking_p1 = add_seconds_after(tracking_p1, "Game Clock", "Game_Clock_Seconds")
tracking_p2 = add_seconds_after(tracking_p2, "Game Clock", "Game_Clock_Seconds")
tracking_p3 = add_seconds_after(tracking_p3, "Game Clock", "Game_Clock_Seconds")

### Distribution of Shot Types in the dataset

In [ ]:
shot_types = (
    events.filter(pl.col("Event").is_in(["Shot", "Goal"]))
    .group_by("Detail_1")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("pct"))
    .sort("count", descending=True)
)

(
    GT(shot_types)
    .tab_header(title="Distribution of shot types in the dataset")
    .cols_label(Detail_1="Shot Type", pct="% of Shots")
    .fmt_percent("pct", decimals=1)
    .cols_hide("count")
    .cols_width({"Detail_1": "150px", "pct": "110px"})
)


### Distribution of Shot Outcomes in the dataset

In [ ]:
events.filter(pl.col("Event").is_in(["Shot", "Goal"])).select(
    "Event", "Detail_1", "Detail_2", "Detail_3", "Detail_4"
)

In [ ]:
events.filter(pl.col("Event").is_in(["Shot", "Goal"])).get_column(
    "Detail_2"
).unique().sort()

In [ ]:
descriptions = {
    "Saved": "The shot is on target and saved by the goaltender.",
    "Goal": "The shot is on target and not saved by the goaltender, resulting in a goal.",
    "Blocked": "The shot is blocked by a skater from the defending team.",
    "Missed": "The shot is not on target.",
}

shot_outcomes = (
    events.filter(pl.col("Event").is_in(["Shot", "Goal"]))
    .with_columns(
        pl.when(pl.col("Event") == "Goal")
        .then(pl.lit("Goal"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "On Net"))
        .then(pl.lit("Saved"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Blocked"))
        .then(pl.lit("Blocked"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Missed"))
        .then(pl.lit("Missed"))
        .alias("Outcome")
    )
    .group_by("Outcome")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("pct"))
    .with_columns(pl.col("Outcome").replace(descriptions).alias("Description"))
    .select("Outcome", "Description", "pct")
    .sort("pct", descending=True)
)

(
    GT(shot_outcomes)
    .tab_header(title="Distribution of shot outcomes in the dataset")
    .cols_label(
        Outcome="Shot Outcome",
        Description="Description",
        pct="% of Shots",
    )
    .fmt_percent("pct", decimals=1)
    .cols_width({"Outcome": "130px", "Description": "420px", "pct": "110px"})
)


### Feature-Based Expected Goals Model

#### Visualize Shot Location

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Arc, Circle, FancyBboxPatch

# Prepare shot data with outcome labels, normalized coordinates, and shot angle
shot_data = (
    events
    .filter(
        pl.col("Event").is_in(["Shot", "Goal"]),
        pl.col("Period") == 1,
    )
    .with_columns(
        pl.when(pl.col("Event") == "Goal")
        .then(pl.lit("Goal"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "On Net"))
        .then(pl.lit("Saved"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Blocked"))
        .then(pl.lit("Blocked"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Missed"))
        .then(pl.lit("Missed"))
        .alias("Outcome")
    )
    # Normalize all shots to the right side
    .with_columns(
        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("X_Coordinate"))
        .otherwise(pl.col("X_Coordinate"))
        .alias("X_Coordinate"),

        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("Y_Coordinate"))
        .otherwise(pl.col("Y_Coordinate"))
        .alias("Y_Coordinate"),
    )
    # Compute shot angle: 0° = straight on, 90° = side of net
    .with_columns(
        (
            pl.arctan2(pl.col("Y_Coordinate"), 89 - pl.col("X_Coordinate"))
            .abs()
            * (180 / 3.141592653589793)
        ).alias("Shot_Angle")
    )
    .select("X_Coordinate", "Y_Coordinate", "Outcome", "Shot_Angle")
)

outcome_markers = {
    "Goal":    {"marker": "*", "size": 250},
    "Saved":   {"marker": "o", "size":  90},
    "Blocked": {"marker": "s", "size":  90},
    "Missed":  {"marker": "^", "size":  90},
}

cmap = plt.cm.plasma
angles = shot_data.get_column("Shot_Angle").to_list()
norm = mcolors.Normalize(vmin=min(angles), vmax=max(angles))

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor("#e8f4fb")

# --- Rink ---
ax.add_patch(FancyBboxPatch(
    (-100, -42.5), 200, 85,
    boxstyle="round,pad=0,rounding_size=28",
    linewidth=2, edgecolor="black", facecolor="white", zorder=1,
))
for x, color, lw in [(0, "#c0392b", 3.0), (25, "#2980b9", 3.0), (89, "#c0392b", 1.5)]:
    ax.plot([x, x], [-42.5, 42.5], color=color, linewidth=lw, zorder=2)
for y in [22, -22]:
    ax.add_patch(Circle((69, y), 15, fill=False, color="#c0392b", linewidth=1.5, zorder=2))
ax.add_patch(Arc((89, 0), 12, 12, theta1=270, theta2=90, color="#2980b9", linewidth=1.5, zorder=2))

# --- Plot shots: color = angle, shape = outcome ---
for outcome, style in outcome_markers.items():
    subset = shot_data.filter(pl.col("Outcome") == outcome)
    ax.scatter(
        subset.get_column("X_Coordinate").to_list(),
        subset.get_column("Y_Coordinate").to_list(),
        c=subset.get_column("Shot_Angle").to_list(),
        cmap=cmap, norm=norm,
        marker=style["marker"], s=style["size"],
        zorder=5, edgecolors="black", linewidths=0.5, alpha=0.9,
    )

# --- Colorbar ---
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("Shot Angle (°)", fontsize=11)

# --- Legend for shapes ---
handles = [
    plt.scatter([], [], marker=s["marker"], s=s["size"], c="gray",
                edgecolors="black", linewidths=0.5, label=o)
    for o, s in outcome_markers.items()
]
ax.legend(handles=handles, loc="upper left", fontsize=10, frameon=False, title="Outcome")

ax.set_xlim(0, 105)
ax.set_ylim(-47, 55)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Shot Locations — Period 1\nColor = Shot Angle, Shape = Outcome", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


#### Shot Location

In [ ]:
shot_data.with_columns(
    (
        pl.arctan2(pl.col("Y_Coordinate"), pl.col("X_Coordinate") - 89)
        .abs()
        * (180 / 3.141592653589793)
    ).alias("Shot_Angle")
)

In [ ]:

(
    events
    .filter(pl.col("Event").is_in(["Shot", "Goal"]))
    # Normalize to right side
    .with_columns(
        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("X_Coordinate"))
        .otherwise(pl.col("X_Coordinate"))
        .alias("X_Coordinate"),

        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("Y_Coordinate"))
        .otherwise(pl.col("Y_Coordinate"))
        .alias("Y_Coordinate"),
    )
    # Shot angle (degrees) and meridian distance
    .with_columns(
        (
            pl.arctan2(pl.col("Y_Coordinate"), 89 - pl.col("X_Coordinate"))
            .abs()
            * (180 / math.pi)
        ).alias("Shot_Angle"),
        pl.col("Y_Coordinate").abs().alias("d_m"),
    )
    # Distance via trigonometry: d = d_m / sin(θ), fallback to (89 - x) when Y = 0
    .with_columns(
        pl.when(pl.col("d_m") == 0)
        .then(89 - pl.col("X_Coordinate"))
        .otherwise(
            pl.col("d_m") / (pl.col("Shot_Angle") * math.pi / 180).sin()
        )
        .alias("d_trig"),
        # Euclidean distance as sanity check
        ((89 - pl.col("X_Coordinate")).pow(2) + pl.col("Y_Coordinate").pow(2))
        .sqrt()
        .alias("d_euclidean"),
    )
    .select("X_Coordinate", "Y_Coordinate", "Shot_Angle", "d_m", "d_trig", "d_euclidean")
)

Find distance measures and shot angle, specified in 2.4.1.1 Shot Location

#### Shooter Motion

In [3]:
# Goal: the shooter's *speed* (magnitude of velocity) at the instant they shot.
# Tracking gives position over time; events tell us who/when/where each shot was.
#
# Tracking runs at ~30 fps: the "Image Id" suffix is a per-period frame counter that
# ticks ~30 times per game-clock second (e.g. "..._066486" -> "..._066487"). Frames
# occasionally drop, so velocity uses the *actual* frame-index gap for dt, never an
# assumed constant. We smooth with a small central difference to damp tracking jitter.

FPS = 30                  # tracking frame rate (frames per second)
SMOOTH_K = 3              # central-difference half-window (~0.1 s each side)
FT_PER_S_TO_MPH = 0.6818  # 1 ft/s = 0.6818 mph
MAX_SPEED_MPH = 30.0      # human skaters peak ~25 mph; above this is a tracking glitch
MAX_SPEED_FPS = MAX_SPEED_MPH / FT_PER_S_TO_MPH

# One table of every tracked player-frame across the three periods.
tracking = pl.concat([tracking_p1, tracking_p2, tracking_p3])

players = (
    tracking
    .filter(pl.col("Player or Puck") == "Player")
    .with_columns(
        # Numeric frame index from the Image Id suffix, and jersey as a string for matching.
        pl.col("Image Id").str.split("_").list.last().cast(pl.Int64).alias("Frame"),
        pl.col("Player Jersey Number").cast(pl.String).alias("Jersey"),
    )
    # Drop rows whose jersey couldn't be read: they'd all collapse into one bogus "null"
    # track, and differencing positions of *different* players invents huge fake speeds.
    .filter(pl.col("Jersey").is_not_null())
    # Order each player's frames in time so shift() walks consecutive samples.
    .sort(["Period", "Team", "Jersey", "Frame"])
    .with_columns(
        # Central difference over +/- SMOOTH_K frames, partitioned per player-period.
        (pl.col("Rink Location X (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location X (Feet)").shift(SMOOTH_K))
            .over(["Period", "Team", "Jersey"]).alias("dx"),
        (pl.col("Rink Location Y (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location Y (Feet)").shift(SMOOTH_K))
            .over(["Period", "Team", "Jersey"]).alias("dy"),
        (pl.col("Frame").shift(-SMOOTH_K) - pl.col("Frame").shift(SMOOTH_K))
            .over(["Period", "Team", "Jersey"]).alias("dframe"),
    )
    .with_columns(
        # dt = frames / fps (seconds); speed = displacement / dt, in feet per second.
        ((pl.col("dx") ** 2 + pl.col("dy") ** 2).sqrt() / (pl.col("dframe") / FPS)).alias("speed_fps")
    )
    .with_columns(
        # Null out physically impossible speeds (tracking ID-swaps / position jumps) so
        # they don't poison the per-shot pick or its fallback median.
        pl.when(pl.col("speed_fps") <= MAX_SPEED_FPS).then(pl.col("speed_fps")).otherwise(None).alias("speed_fps")
    )
    .with_columns(
        (pl.col("speed_fps") * FT_PER_S_TO_MPH).alias("speed_mph")
    )
)

players.select(
    "Period", "Team", "Jersey", "Frame", "Game_Clock_Seconds",
    "Rink Location X (Feet)", "Rink Location Y (Feet)", "speed_fps", "speed_mph",
)


Period,Team,Jersey,Frame,Game_Clock_Seconds,Rink Location X (Feet),Rink Location Y (Feet),speed_fps,speed_mph
i64,str,str,i64,i64,f64,f64,f64,f64
1,"""Away""","""14""",71948,1053,67.427,21.172,null,null
1,"""Away""","""14""",71949,1052,67.304,21.257,null,null
1,"""Away""","""14""",71950,1052,67.365,21.779,null,null
1,"""Away""","""14""",71951,1052,67.723,22.361,14.579383,9.940223
1,"""Away""","""14""",71952,1052,67.885,22.909,17.321562,11.809841
…,…,…,…,…,…,…,…,…
3,"""Home""","""Go""",325726,21,85.497,-1.456,9.326003,6.358469
3,"""Home""","""Go""",325727,21,85.204,-2.158,0.517046,0.352522
3,"""Home""","""Go""",325728,21,84.913,-2.747,null,null


In [4]:
# Match each Shot/Goal event to the tracking frame where the puck was released, then
# read the shooter's speed there.
#
# Matching keys: events Player_Id is a jersey number, but jerseys repeat across teams,
# so we match on Period + Team + Jersey. Event "Team" is a name (e.g. "Team A");
# tracking "Team" is Home/Away -> map via Home_Team/Away_Team.
#
# Pinning the release: the event stores the release X/Y. Within +/-1 game-clock second
# of the event we pick the shooter frame whose tracked position is closest to that
# release point. Event and tracking share the same rink frame here, but to be safe we
# also test the sign-flipped coords (-X,-Y) and keep whichever is closer.

shots = (
    events.filter(pl.col("Event").is_in(["Shot", "Goal"]))
    .with_columns(
        pl.when(pl.col("Team") == pl.col("Home_Team")).then(pl.lit("Home"))
          .otherwise(pl.lit("Away")).alias("Track_Team")
    )
    .with_row_index("shot_id")
)

# Candidate shooter frames: same player, within the event second (+/-1s), valid position.
cand = (
    shots.join(
        players.select([
            "Period", "Team", "Jersey", "Game_Clock_Seconds", "Frame",
            "Rink Location X (Feet)", "Rink Location Y (Feet)", "speed_fps",
        ]),
        left_on=["Period", "Track_Team", "Player_Id"],
        right_on=["Period", "Team", "Jersey"], how="left",
    )
    .filter((pl.col("Game_Clock_Seconds") - pl.col("Clock_Seconds")).abs() <= 1)
    .filter(pl.col("Rink Location X (Feet)").is_not_null()
            & pl.col("Rink Location Y (Feet)").is_not_null())
    .with_columns(
        ((pl.col("Rink Location X (Feet)") - pl.col("X_Coordinate")) ** 2
         + (pl.col("Rink Location Y (Feet)") - pl.col("Y_Coordinate")) ** 2).sqrt().alias("d_direct"),
        ((pl.col("Rink Location X (Feet)") + pl.col("X_Coordinate")) ** 2
         + (pl.col("Rink Location Y (Feet)") + pl.col("Y_Coordinate")) ** 2).sqrt().alias("d_flip"),
    )
    .with_columns(
        pl.min_horizontal("d_direct", "d_flip").alias("match_dist"),
        (pl.col("d_flip") < pl.col("d_direct")).alias("used_flip"),
    )
)

# Release frame = the candidate closest to the release point (nulls_last so a real
# match is never beaten by a missing distance).
release = (
    cand.sort("match_dist", nulls_last=True)
    .group_by("shot_id", maintain_order=True)
    .first()
    .select(["shot_id", "Frame", "match_dist", "used_flip", "speed_fps"])
)

# Fallback: representative speed over the whole event-second window.
window_med = cand.group_by("shot_id").agg(
    pl.col("speed_fps").median().alias("speed_fps_window_median"),
    pl.len().alias("n_candidates"),
)

# A match is trusted when the closest frame is within ~10 ft and has a defined speed;
# otherwise fall back to the window median (flagged), or mark no tracking at all.
MAX_MATCH_DIST = 10.0
shot_speeds = (
    shots.join(release, on="shot_id", how="left")
    .join(window_med, on="shot_id", how="left")
    .with_columns(
        pl.when((pl.col("match_dist") <= MAX_MATCH_DIST) & pl.col("speed_fps").is_not_null())
          .then(pl.col("speed_fps"))
          .otherwise(pl.col("speed_fps_window_median"))
          .alias("shooter_speed_fps"),
        pl.when((pl.col("match_dist") <= MAX_MATCH_DIST) & pl.col("speed_fps").is_not_null())
          .then(pl.lit("matched"))
          .when(pl.col("speed_fps_window_median").is_not_null())
          .then(pl.lit("fallback_window_median"))
          .otherwise(pl.lit("no_tracking"))
          .alias("match_quality"),
    )
    .with_columns((pl.col("shooter_speed_fps") * 0.6818).alias("shooter_speed_mph"))
    .sort(["Period", "Clock_Seconds"], descending=[False, True])
)

print(shot_speeds["match_quality"].value_counts())
shot_speeds.select(
    "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
    "Frame", "match_dist", "used_flip", "shooter_speed_fps", "shooter_speed_mph", "match_quality",
).head()


shape: (2, 2)
┌────────────────────────┬───────┐
│ match_quality          ┆ count │
│ ---                    ┆ ---   │
│ str                    ┆ u32   │
╞════════════════════════╪═══════╡
│ fallback_window_median ┆ 2     │
│ matched                ┆ 117   │
└────────────────────────┴───────┘


Period,Clock,Team,Player_Id,Event,Detail_1,Frame,match_dist,used_flip,shooter_speed_fps,shooter_speed_mph,match_quality
i64,str,str,str,str,str,i64,f64,bool,f64,f64,str
1,"""19:54""","""Team D""","""42""","""Shot""","""Wristshot""",65668,1.257159,false,22.062466,15.04219,"""matched"""
1,"""19:26""","""Team A""","""29""","""Shot""","""Wristshot""",66490,3.945989,false,20.065956,13.680969,"""matched"""
1,"""19:16""","""Team A""","""72""","""Shot""","""Wristshot""",66824,4.759513,false,4.372238,2.980992,"""matched"""
1,"""19:15""","""Team A""","""29""","""Shot""","""Deflection""",66861,1.051468,false,3.771091,2.57113,"""matched"""
1,"""19:13""","""Team A""","""63""","""Shot""","""Wristshot""",66873,0.13366,false,4.189063,2.856103,"""matched"""


In [ ]:
# Shooter speed at the moment of each shot, in ft/s and mph.
shot_speed_table = shot_speeds.select(
    "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
    "shooter_speed_fps", "shooter_speed_mph", "match_quality",
)

(
    GT(shot_speed_table)
    .tab_header(
        title="Shooter speed at the moment of the shot",
        subtitle="Magnitude of the shooter's velocity from tracking, pinned to the release frame",
    )
    .cols_label(
        Player_Id="Jersey",
        Detail_1="Shot Type",
        shooter_speed_fps="Speed (ft/s)",
        shooter_speed_mph="Speed (mph)",
        match_quality="Match",
    )
    .fmt_number(["shooter_speed_fps", "shooter_speed_mph"], decimals=1)
    .cols_width({
        "Period": "70px", "Clock": "70px", "Team": "90px", "Player_Id": "70px",
        "Event": "70px", "Detail_1": "110px",
        "shooter_speed_fps": "110px", "shooter_speed_mph": "110px", "match_quality": "150px",
    })
)


#### Pressure

In [ ]:
# Pressure: distance from the shot location to the closest and second-closest
# *defending skater*. Following the cited paper, a continuous distance is more
# descriptive than counting defenders within 6 ft, and adding the second-closest
# defender captures pressure from multiple defenders.
#
# Goalies (jersey "Go") are excluded: a goalie sits in the crease on nearly every
# shot, so it would dominate the "closest defender" distance without reflecting
# real defensive pressure.
#
# We reuse the release Frame matched in the shooter-speed step: at that frame we take
# every opposing skater's tracked position and measure straight-line distance to the
# shot location. "used_flip" (from the same matching) tells us whether the event's
# (X, Y) had to be negated to line up with the tracking frame for that shot, so we
# apply the same flip to the shot coordinates here.
#
# Note: tracking coverage is partial -- many frames carry fewer than 5 skaters per team
# (see n_defenders). Coverage concentrates near the play, so the closest defenders (the
# ones this feature needs) are the most reliably tracked; n_defenders records how many
# defenders were actually visible at each shot's release frame.

# Defending skaters only: opposite tracking team, real jersey numbers, no goalies.
defenders = (
    players
    .filter(pl.col("Jersey") != "Go")
    .select([
        "Period", "Team", "Frame",
        "Rink Location X (Feet)", "Rink Location Y (Feet)",
    ])
)

# Per shot: the defending team, the release frame, and the shot location in tracking
# coordinates (negated when the release match used the flip).
shot_frames = (
    shots
    .join(release.select(["shot_id", "Frame", "used_flip"]), on="shot_id", how="left")
    .with_columns(
        pl.when(pl.col("Track_Team") == "Home").then(pl.lit("Away"))
          .otherwise(pl.lit("Home")).alias("Def_Team"),
        pl.when(pl.col("used_flip")).then(-pl.col("X_Coordinate"))
          .otherwise(pl.col("X_Coordinate")).alias("Shot_X"),
        pl.when(pl.col("used_flip")).then(-pl.col("Y_Coordinate"))
          .otherwise(pl.col("Y_Coordinate")).alias("Shot_Y"),
    )
    .select(["shot_id", "Period", "Def_Team", "Frame", "Shot_X", "Shot_Y"])
)

# Join each shot to every defending skater in its release frame; measure distance.
pressure = (
    shot_frames
    .join(
        defenders,
        left_on=["Period", "Def_Team", "Frame"],
        right_on=["Period", "Team", "Frame"],
        how="left",
    )
    .with_columns(
        ((pl.col("Rink Location X (Feet)") - pl.col("Shot_X")) ** 2
         + (pl.col("Rink Location Y (Feet)") - pl.col("Shot_Y")) ** 2)
        .sqrt().alias("def_dist")
    )
    .group_by("shot_id", maintain_order=True)
    .agg(
        pl.col("def_dist").sort(nulls_last=True).alias("def_dists"),
        pl.col("def_dist").is_not_null().sum().alias("n_defenders"),
    )
    .with_columns(
        pl.col("def_dists").list.get(0, null_on_oob=True).alias("closest_def_dist"),
        pl.col("def_dists").list.get(1, null_on_oob=True).alias("second_closest_def_dist"),
    )
    .drop("def_dists")
)

# Attach to the shots table for use as expected-goals model features.
shot_pressure = shots.join(pressure, on="shot_id", how="left").select(
    "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
    "n_defenders", "closest_def_dist", "second_closest_def_dist",
)


shot_pressure

#### Pre-Shot Movement

In [ ]:
# Pre-shot movement features, focused on lateral puck movement (following the cited
# paper / Stathletes "Opportunity Analysis"):
#
#   1. time_since_meridian  -- seconds since the puck last crossed the meridian
#   2. meridian_cross_dist  -- where it crossed, as distance to the goal line
#   3. avg_puck_speed       -- average puck speed in the 1 s before the shot
#
# The "meridian" is the Y = 0 line running net-to-net down the center of the ice; the
# puck crosses it when its Y coordinate changes sign (lateral / east-west movement,
# which forces the goalie across and raises shot danger). Features 1 and 2 are only
# defined when the crossing happened within 5 s before the shot.
#
# We reuse the per-shot release Frame and "used_flip" orientation matched in the
# shooter-speed step. Puck X/Y are normalized to attack-right (net at +89) using each
# shot's orientation so the goal-line distance is signed consistently: a crossing in
# front of the net is positive, one behind the goal line (X > 89) is negative.

# The puck moves far faster than skaters, so the 30 mph skater cap doesn't apply; we
# only drop physically impossible jumps (tracking glitches) above ~110 mph.
MAX_PUCK_SPEED_MPH = 110.0
MAX_PUCK_SPEED_FPS = MAX_PUCK_SPEED_MPH / FT_PER_S_TO_MPH

# Clean puck track with per-frame speed (same central-difference scheme as players,
# but partitioned by Period only -- the puck is a single object). ~95% of puck rows
# carry a valid position; the rest are dropped.
puck = (
    tracking
    .filter(pl.col("Player or Puck") == "Puck")
    .filter(pl.col("Rink Location X (Feet)").is_not_null())
    .with_columns(
        pl.col("Image Id").str.split("_").list.last().cast(pl.Int64).alias("Frame")
    )
    .sort(["Period", "Frame"])
    .with_columns(
        (pl.col("Rink Location X (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location X (Feet)").shift(SMOOTH_K)).over("Period").alias("dx"),
        (pl.col("Rink Location Y (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location Y (Feet)").shift(SMOOTH_K)).over("Period").alias("dy"),
        (pl.col("Frame").shift(-SMOOTH_K) - pl.col("Frame").shift(SMOOTH_K)).over("Period").alias("dframe"),
    )
    .with_columns(
        ((pl.col("dx") ** 2 + pl.col("dy") ** 2).sqrt() / (pl.col("dframe") / FPS)).alias("puck_speed_fps")
    )
    .with_columns(
        pl.when(pl.col("puck_speed_fps") <= MAX_PUCK_SPEED_FPS).then(pl.col("puck_speed_fps")).otherwise(None).alias("puck_speed_fps")
    )
    .select(["Period", "Frame", "Rink Location X (Feet)", "Rink Location Y (Feet)", "puck_speed_fps"])
    .rename({"Rink Location X (Feet)": "puck_x", "Rink Location Y (Feet)": "puck_y"})
)

# Per-shot orientation: the shot location in tracking coords (negated when the release
# match used the flip) tells us which goal line the team attacks (+89 if X > 0).
shot_orient = (
    shots.join(release.select(["shot_id", "Frame", "used_flip"]), on="shot_id", how="left")
    .with_columns(
        pl.when(pl.col("used_flip")).then(-pl.col("X_Coordinate")).otherwise(pl.col("X_Coordinate")).alias("Shot_X_track")
    )
    .with_columns((pl.col("Shot_X_track") > 0).alias("attack_right"))
    .select(["shot_id", "Period", "Frame", "attack_right"])
    .rename({"Frame": "shot_frame"})
)

# Puck samples in the same period within 5 s (150 frames) before each shot, with
# coordinates normalized to attack-right. (The meridian sign-change is invariant to the
# flip; normalizing X only matters for the goal-line distance.)
window = (
    shot_orient.join(puck, on="Period", how="left")
    .filter((pl.col("shot_frame") - pl.col("Frame")).is_between(0, 5 * FPS))
    .with_columns(
        pl.when(pl.col("attack_right")).then(pl.col("puck_x")).otherwise(-pl.col("puck_x")).alias("px"),
        pl.when(pl.col("attack_right")).then(pl.col("puck_y")).otherwise(-pl.col("puck_y")).alias("py"),
    )
    .sort(["shot_id", "Frame"])
)

# Feature 3: average puck speed over the 1 s (30 frames) before the shot.
avg_speed = (
    window.filter((pl.col("shot_frame") - pl.col("Frame")).is_between(0, 1 * FPS))
    .group_by("shot_id")
    .agg(pl.col("puck_speed_fps").mean().alias("avg_puck_speed_fps"))
    .with_columns((pl.col("avg_puck_speed_fps") * FT_PER_S_TO_MPH).alias("avg_puck_speed_mph"))
)

# Features 1 & 2: the most recent meridian crossing within the 5 s window. A crossing is
# a sign change in py between consecutive frames; we linearly interpolate the X where
# py = 0, then keep the latest crossing (largest Frame) before the shot.
crossings = (
    window.with_columns(
        pl.col("px").shift(1).over("shot_id").alias("px_prev"),
        pl.col("py").shift(1).over("shot_id").alias("py_prev"),
    )
    .filter(pl.col("py_prev").is_not_null() & (pl.col("py_prev") * pl.col("py") < 0))
    .with_columns(
        (pl.col("px_prev") + (pl.col("px") - pl.col("px_prev")) * (0 - pl.col("py_prev")) / (pl.col("py") - pl.col("py_prev"))).alias("x_cross")
    )
    .sort(["shot_id", "Frame"])
    .group_by("shot_id", maintain_order=True)
    .last()
    .with_columns(
        ((pl.col("shot_frame") - pl.col("Frame")) / FPS).alias("time_since_meridian"),
        (89 - pl.col("x_cross")).alias("meridian_cross_dist"),  # negative if crossed behind the net
    )
    .select(["shot_id", "time_since_meridian", "meridian_cross_dist"])
)

# Attach to the shots table; nulls mark shots with no meridian crossing in the prior 5 s.
pre_shot_movement = (
    shots.join(crossings, on="shot_id", how="left")
    .join(avg_speed, on="shot_id", how="left")
    .sort(["Period", "Clock_Seconds"], descending=[False, True])
    .select(
        "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
        "time_since_meridian", "meridian_cross_dist", "avg_puck_speed_fps", "avg_puck_speed_mph",
    )
)

(
    GT(pre_shot_movement)
    .tab_header(
        title="Pre-shot puck movement",
        subtitle="Lateral movement (meridian crossings) and puck speed in the lead-up to each shot",
    )
    .cols_label(
        Player_Id="Jersey",
        Detail_1="Shot Type",
        time_since_meridian="Time Since Meridian (s)",
        meridian_cross_dist="Cross Dist to Goal Line (ft)",
        avg_puck_speed_fps="Puck Speed (ft/s)",
        avg_puck_speed_mph="Puck Speed (mph)",
    )
    .fmt_number(
        ["time_since_meridian", "meridian_cross_dist", "avg_puck_speed_fps", "avg_puck_speed_mph"],
        decimals=1,
    )
    .sub_missing(columns=["time_since_meridian", "meridian_cross_dist", "avg_puck_speed_fps", "avg_puck_speed_mph"], missing_text="—")
    .cols_width({
        "Period": "60px", "Clock": "60px", "Team": "80px", "Player_Id": "60px",
        "Event": "60px", "Detail_1": "100px",
        "time_since_meridian": "120px", "meridian_cross_dist": "130px",
        "avg_puck_speed_fps": "110px", "avg_puck_speed_mph": "110px",
    })
)


In [ ]:
MAX_PUCK_SPEED_MPH = 110.0
MAX_PUCK_SPEED_FPS = MAX_PUCK_SPEED_MPH / FT_PER_S_TO_MPH

puck = (
    tracking
    .filter(pl.col("Player or Puck") == "Puck")
    .filter(pl.col("Rink Location X (Feet)").is_not_null())
    .with_columns(
        pl.col("Image Id").str.split("_").list.last().cast(pl.Int64).alias("Frame")
    )
    .sort(["Period", "Frame"])
    .with_columns(
        (pl.col("Rink Location X (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location X (Feet)").shift(SMOOTH_K)).over("Period").alias("dx"),
        (pl.col("Rink Location Y (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location Y (Feet)").shift(SMOOTH_K)).over("Period").alias("dy"),
        (pl.col("Frame").shift(-SMOOTH_K) - pl.col("Frame").shift(SMOOTH_K)).over("Period").alias("dframe"),
    )
    .with_columns(
        ((pl.col("dx") ** 2 + pl.col("dy") ** 2).sqrt() / (pl.col("dframe") / FPS)).alias("puck_speed_fps")
    )
    .with_columns(
        pl.when(pl.col("puck_speed_fps") <= MAX_PUCK_SPEED_FPS).then(pl.col("puck_speed_fps")).otherwise(None).alias("puck_speed_fps")
    )
    .select(["Period", "Frame", "Rink Location X (Feet)", "Rink Location Y (Feet)", "puck_speed_fps"])
    .rename({"Rink Location X (Feet)": "puck_x", "Rink Location Y (Feet)": "puck_y"})
)


puck

In [ ]:
shot_orient = (
    shots.join(release.select(["shot_id", "Frame", "used_flip"]), on="shot_id", how="left")
    .with_columns(
        pl.when(pl.col("used_flip")).then(-pl.col("X_Coordinate")).otherwise(pl.col("X_Coordinate")).alias("Shot_X_track")
    )
    .with_columns((pl.col("Shot_X_track") > 0).alias("attack_right"))
    .select(["shot_id", "Period", "Frame", "attack_right"])
    .rename({"Frame": "shot_frame"})
)
shot_orient

In [ ]:
window = (
    shot_orient.join(puck, on="Period", how="left")
    .filter((pl.col("shot_frame") - pl.col("Frame")).is_between(0, 5 * FPS))
    .with_columns(
        pl.when(pl.col("attack_right")).then(pl.col("puck_x")).otherwise(-pl.col("puck_x")).alias("px"),
        pl.when(pl.col("attack_right")).then(pl.col("puck_y")).otherwise(-pl.col("puck_y")).alias("py"),
    )
    .sort(["shot_id", "Frame"])
)

window

In [ ]:
avg_speed = (
    window.filter((pl.col("shot_frame") - pl.col("Frame")).is_between(0, 1 * FPS))
    .group_by("shot_id")
    .agg(pl.col("puck_speed_fps").mean().alias("avg_puck_speed_fps"))
    .with_columns((pl.col("avg_puck_speed_fps") * FT_PER_S_TO_MPH).alias("avg_puck_speed_mph"))
)

avg_speed

In [ ]:
crossings = (
    window.with_columns(
        pl.col("px").shift(1).over("shot_id").alias("px_prev"),
        pl.col("py").shift(1).over("shot_id").alias("py_prev"),
    )
    .filter(pl.col("py_prev").is_not_null() & (pl.col("py_prev") * pl.col("py") < 0))
    .with_columns(
        (pl.col("px_prev") + (pl.col("px") - pl.col("px_prev")) * (0 - pl.col("py_prev")) / (pl.col("py") - pl.col("py_prev"))).alias("x_cross")
    )
    .sort(["shot_id", "Frame"])
    .group_by("shot_id", maintain_order=True)
    .last()
    .with_columns(
        ((pl.col("shot_frame") - pl.col("Frame")) / FPS).alias("time_since_meridian"),
        (89 - pl.col("x_cross")).alias("meridian_cross_dist"),  # negative if crossed behind the net
    )
    .select(["shot_id", "time_since_meridian", "meridian_cross_dist"])
)

crossings

#### Traffic

In [5]:
# Goal-face occlusion: model traffic by projecting each skater onto the goal line
# via a straight line from the shooter, then summing the Gaussian-weighted coverage
# of the 6-ft goal face (y ∈ [−3, 3]).
#
# For each (shot, skater) pair:
#   1. Extend the shooter→skater line to the goal line (x = goal_x).
#   2. Center a Gaussian (σ = 1.5 ft) at the y-intercept on the goal line.
#   3. Integrate from −3 to +3 ft using the normal CDF via math.erf.
#   4. Sum across all skaters → total_occlusion for the shot.
#
# Skaters from *both* teams can occlude (teammates screen too). Only the shooter
# and goalies are excluded.

import numpy as np

SIGMA = 1.5  # Gaussian standard deviation (feet)
GOAL_HALF = 3.0  # half-width of goal face (feet)


def _normal_cdf_batched(x: np.ndarray) -> np.ndarray:
    """Vectorised normal CDF using math.erf identity: Φ(z) = 0.5*(1+erf(z/√2))."""
    return 0.5 * (1.0 + np.vectorize(math.erf)(x / math.sqrt(2)))


# --- Per-shot info in tracking coordinates (reuses release match) ---
shot_info = (
    shots.join(release.select(["shot_id", "Frame", "used_flip"]), on="shot_id", how="left")
    .with_columns(
        # Shot location in tracking coords
        pl.when(pl.col("used_flip")).then(-pl.col("X_Coordinate")).otherwise(pl.col("X_Coordinate")).alias("Shot_X"),
        pl.when(pl.col("used_flip")).then(-pl.col("Y_Coordinate")).otherwise(pl.col("Y_Coordinate")).alias("Shot_Y"),
    )
    .with_columns(
        # Goal line: +89 if attacking right, −89 if attacking left
        pl.when(pl.col("Shot_X") > 0).then(pl.lit(89.0)).otherwise(pl.lit(-89.0)).alias("goal_x"),
    )
    .select(["shot_id", "Period", "Frame", "Track_Team", "Player_Id", "Shot_X", "Shot_Y", "goal_x"])
)

# --- All non-goalie skaters at each shot's release frame ---
skaters_at_frame = (
    players
    .filter(pl.col("Jersey") != "Go")
    .select(["Period", "Team", "Jersey", "Frame",
             "Rink Location X (Feet)", "Rink Location Y (Feet)"])
    .rename({"Rink Location X (Feet)": "sk_x", "Rink Location Y (Feet)": "sk_y"})
)

# Join each shot with every skater present at the release frame, then drop the shooter.
occlusion_pairs = (
    shot_info.join(
        skaters_at_frame,
        left_on=["Period", "Frame"],
        right_on=["Period", "Frame"],
        how="left",
    )
    # Exclude the shooter (same team in tracking namespace + same jersey)
    .filter(
        ~((pl.col("Track_Team") == pl.col("Team")) & (pl.col("Player_Id") == pl.col("Jersey")))
    )
    .filter(pl.col("sk_x").is_not_null() & pl.col("sk_y").is_not_null())
    # Compute y-intercept on the goal line
    .with_columns(
        (pl.col("Shot_Y")
         + (pl.col("sk_y") - pl.col("Shot_Y"))
           * (pl.col("goal_x") - pl.col("Shot_X"))
           / (pl.col("sk_x") - pl.col("Shot_X"))
        ).alias("y_goal")
    )
    # Drop degenerate cases: skater at same x as shooter, or intercept is undefined
    .filter(pl.col("y_goal").is_not_null() & pl.col("y_goal").is_finite())
)

# Compute Gaussian contribution per skater via map_batches
occlusion_pairs = occlusion_pairs.with_columns(
    pl.struct(["y_goal"]).map_batches(
        lambda s: pl.Series(
            _normal_cdf_batched((GOAL_HALF - s.struct.field("y_goal").to_numpy()) / SIGMA)
            - _normal_cdf_batched((-GOAL_HALF - s.struct.field("y_goal").to_numpy()) / SIGMA)
        ),
        return_dtype=pl.Float64,
    ).alias("occlusion_contribution")
)

# Sum per shot
total_occlusion = (
    occlusion_pairs
    .group_by("shot_id", maintain_order=True)
    .agg(pl.col("occlusion_contribution").sum().alias("total_occlusion"))
)

total_occlusion

shot_id,total_occlusion
u32,f64
0,7.0610e-14
1,0.956973
2,0.914772
3,1.660416
4,2.307789
…,…
114,1.115928
115,0.922711
116,2.375254


In [ ]:
# Display the occlusion feature alongside shot context.
shot_occlusion = (
    shots.join(total_occlusion, on="shot_id", how="left")
    .select(
        "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
        "X_Coordinate", "Y_Coordinate", "total_occlusion",
    )
)

(
    GT(shot_occlusion)
    .tab_header(
        title="Goal-face occlusion (traffic) per shot",
        subtitle="Sum of Gaussian-weighted goal-face coverage from all skaters (σ = 1.5 ft)",
    )
    .cols_label(
        Player_Id="Jersey",
        Detail_1="Shot Type",
        X_Coordinate="X",
        Y_Coordinate="Y",
        total_occlusion="Occlusion",
    )
    .fmt_number("total_occlusion", decimals=3)
    .cols_width({
        "Period": "70px", "Clock": "70px", "Team": "90px", "Player_Id": "70px",
        "Event": "70px", "Detail_1": "110px",
        "X_Coordinate": "60px", "Y_Coordinate": "60px", "total_occlusion": "100px",
    })
)

#### Goaltender Positioning

$$\zeta = \arccos!\left(\frac{s \cdot g}{|s|,|g|}\right)\times\frac{180}{\pi}$$



In [ ]:
# Goaltender positioning at the moment of each shot:
#
#   1. angle_discrepancy (ζ) -- the angle, at the goal center, between the line to the
#      shooter and the line to the goaltender. 0° means the goalie is squared up on the
#      shot's line; larger ζ means the goalie is off-angle and the net is more exposed.
#   2. goaltender_depth (d_g) -- straight-line distance from the goaltender to the goal
#      center, i.e. how far out of the crease the goalie has come.
#
# We reuse the release Frame + "used_flip" matched in the shooter-speed step, exactly as
# the Pressure and Traffic features do: shot coordinates are flipped into tracking space,
# the goal center is (goal_x, 0) with goal_x = +89 / -89 set by which net is attacked,
# and the *defending* team's goalie (jersey "Go") is read at that same frame. Shots with
# no goalie tracked at the release frame get null for both features (left as-is, like
# n_defenders / match_quality elsewhere).

# Per shot: defending team, release frame, shot location in tracking coords, goal center.
shot_goalie_ctx = (
    shots.join(release.select(["shot_id", "Frame", "used_flip"]), on="shot_id", how="left")
    .with_columns(
        pl.when(pl.col("Track_Team") == "Home").then(pl.lit("Away"))
          .otherwise(pl.lit("Home")).alias("Def_Team"),
        pl.when(pl.col("used_flip")).then(-pl.col("X_Coordinate"))
          .otherwise(pl.col("X_Coordinate")).alias("Shot_X"),
        pl.when(pl.col("used_flip")).then(-pl.col("Y_Coordinate"))
          .otherwise(pl.col("Y_Coordinate")).alias("Shot_Y"),
    )
    .with_columns(
        # Goal line: +89 if attacking right, −89 if attacking left.
        pl.when(pl.col("Shot_X") > 0).then(pl.lit(89.0)).otherwise(pl.lit(-89.0)).alias("goal_x"),
    )
    .select(["shot_id", "Period", "Def_Team", "Frame", "Shot_X", "Shot_Y", "goal_x"])
)

# Defending goaltender's tracked position at each shot's release frame.
goalies = (
    players
    .filter(pl.col("Jersey") == "Go")
    .select(["Period", "Team", "Frame",
             "Rink Location X (Feet)", "Rink Location Y (Feet)"])
    .rename({"Rink Location X (Feet)": "goalie_x", "Rink Location Y (Feet)": "goalie_y"})
)

goaltender_positioning = (
    shot_goalie_ctx
    .join(
        goalies,
        left_on=["Period", "Def_Team", "Frame"],
        right_on=["Period", "Team", "Frame"],
        how="left",
    )
    # One goalie per team-frame is expected; guard against duplicate tracks.
    .group_by("shot_id", maintain_order=True)
    .first()
    .with_columns(
        # Vectors from the goal center to the shooter and to the goalie.
        (pl.col("Shot_X") - pl.col("goal_x")).alias("s_x"),
        pl.col("Shot_Y").alias("s_y"),
        (pl.col("goalie_x") - pl.col("goal_x")).alias("g_x"),
        pl.col("goalie_y").alias("g_y"),
    )
    .with_columns(
        (pl.col("s_x") ** 2 + pl.col("s_y") ** 2).sqrt().alias("s_mag"),
        (pl.col("g_x") ** 2 + pl.col("g_y") ** 2).sqrt().alias("g_mag"),
        (pl.col("s_x") * pl.col("g_x") + pl.col("s_y") * pl.col("g_y")).alias("dot"),
    )
    .with_columns(
        # d_g: distance from the goaltender to the goal center.
        pl.col("g_mag").alias("goaltender_depth"),
        # ζ (degrees): undefined when shooter or goalie sits exactly at the goal center.
        pl.when((pl.col("s_mag") > 0) & (pl.col("g_mag") > 0))
          .then(
              (pl.col("dot") / (pl.col("s_mag") * pl.col("g_mag")))
              .clip(-1.0, 1.0).arccos() * (180 / math.pi)
          )
          .otherwise(None)
          .alias("angle_discrepancy"),
    )
    .select(["shot_id", "angle_discrepancy", "goaltender_depth"])
)

# Attach to the shots table for use as expected-goals model features.
shot_goaltender = (
    shots.join(goaltender_positioning, on="shot_id", how="left")
    .sort(["Period", "Clock_Seconds"], descending=[False, True])
    .select(
        "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
        "angle_discrepancy", "goaltender_depth",
    )
)

(
    GT(shot_goaltender)
    .tab_header(
        title="Goaltender positioning per shot",
        subtitle="Angle discrepancy (ζ) and depth (d_g) of the defending goalie at the release frame",
    )
    .cols_label(
        Player_Id="Jersey",
        Detail_1="Shot Type",
        angle_discrepancy="Angle Discrepancy ζ (°)",
        goaltender_depth="Depth d_g (ft)",
    )
    .fmt_number(["angle_discrepancy", "goaltender_depth"], decimals=1)
    .sub_missing(columns=["angle_discrepancy", "goaltender_depth"], missing_text="—")
    .cols_width({
        "Period": "60px", "Clock": "60px", "Team": "80px", "Player_Id": "60px",
        "Event": "60px", "Detail_1": "100px",
        "angle_discrepancy": "150px", "goaltender_depth": "120px",
    })
)

#### XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Prepare the dataset for modeling: select features and target, handle missing values, encode categoricals.
